<a href="https://colab.research.google.com/github/vincentmartin/tp-initiation-llm-student-version/blob/main/TP_initiation_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP d'initiation aux LLM

Dans ce TP, vous allez apprendre les bases de l'IA générative en manipulant et en contrôlant un LLM **open-weight** exécuté localement sur le GPU de Colab : aucun compte payant ni clé d'API n'est nécessaire.

En sortie de ce TP, vous serez capable de :
- Expliquer ce qu'est un token, un tokenizer, un *chat template* et une fenêtre de contexte
- Observer ce que calcule réellement un LLM : une distribution de probabilité sur le token suivant
- Distinguer un modèle *base* d'un modèle *instruct*, et utiliser un modèle de raisonnement (*thinking*)
- Appliquer les techniques de prompt engineering : system prompt, zero-shot, one-shot, few-shot
- Contrôler la génération (température, top-k, top-p, échantillonnage)
- Évaluer des sorties de LLM (ROUGE, LLM-as-a-judge)
- Obtenir des sorties structurées (JSON validé par Pydantic)
- Construire une application : RAG sur un document ou agent avec appel d'outils

**Modèle utilisé** : [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) (Alibaba, licence Apache 2.0). Petit, multilingue, avec un mode raisonnement activable : il tient sans problème sur le GPU T4 gratuit de Colab.

### Instructions pour l'exécution sur Google Colab

Aller dans `Exécution -> Modifier le type d'exécution` puis sélectionner `T4 GPU`.

![Colab GPU](https://raw.githubusercontent.com/vincentmartin/tp-initiation-llm-student-version/main/resources/colab_gpu.png "T4-GPU")

**Pas de GPU disponible sur Colab ?** (quota atteint, heure de pointe) : utilisez [Kaggle Notebooks](https://www.kaggle.com/code), gratuit avec environ 30 h de GPU par semaine (`Settings -> Accelerator -> GPU T4 x2`, et activer `Internet` après vérification du compte par téléphone). Importez ce notebook via `File -> Import Notebook`.

**Assistants IA** : vous pouvez utiliser l'assistant intégré à Colab, mais vous devez être capables d'expliquer **chaque ligne** du code que vous rendez. Les questions d'analyse comptent autant que le code.

## 0. Installation et configuration

Colab fournit déjà `torch` : **ne le réinstallez pas**, cela casserait la compatibilité avec CUDA. Nous mettons simplement à jour les bibliothèques Hugging Face :
- **transformers** : chargement et utilisation des modèles de type Transformer
- **accelerate** : placement automatique du modèle sur le GPU
- **datasets** : téléchargement de jeux de données depuis le Hub Hugging Face
- **evaluate** / **rouge_score** : métriques d'évaluation

In [ ]:
%pip install -q -U transformers accelerate datasets evaluate rouge_score

**Remarque** : si l'import ci-dessous échoue juste après l'installation, redémarrez la session (`Exécution -> Redémarrer la session`) puis relancez les cellules à partir d'ici.

In [ ]:
import gc
import json
import re
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

print(f"torch {torch.__version__} | transformers {transformers.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
    # Le T4 (architecture Turing, compute capability 7.5) ne gère pas nativement le bfloat16 :
    # on utilise float16 sur ce type de GPU, bfloat16 sur les GPU récents (Ampere et +).
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    print("ATTENTION : pas de GPU, l'exécution sera très lente. Vérifiez le type d'exécution.")
    DTYPE = torch.float32

set_seed(42)

## 1. Tokenisation

Un LLM ne manipule pas des caractères ni des mots, mais des **tokens** : des morceaux de mots issus d'un vocabulaire fixe (environ 150 000 tokens pour Qwen3). Le **tokenizer** convertit le texte en une suite d'identifiants entiers (*ids*) et inversement.

Chargeons le tokenizer de notre modèle.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Taille du vocabulaire : {len(tokenizer)}")

**Exercice 1** : en vous aidant de la [documentation des tokenizers](https://huggingface.co/docs/transformers/main_classes/tokenizer) :
- encoder la phrase en une liste d'ids et l'afficher ;
- afficher les tokens correspondant à ces ids (méthode `convert_ids_to_tokens`) ;
- décoder la liste d'ids pour retrouver la phrase.

In [ ]:
sentence = "Que peux-tu me dire sur les LLMs ?"

# VOTRE CODE ICI

Le caractère `Ġ` représente un espace : il fait partie du token qui le suit.

**Exercice 2** : comparons le nombre de tokens nécessaires pour une même phrase en français et en anglais, avec le tokenizer de Qwen3 et celui de `flan-t5`, le modèle utilisé dans les versions précédentes de ce TP (2022). Écrire une fonction `count_tokens(tok, text)` puis compléter le tableau.

In [ ]:
old_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

sentences = {
    "fr": "L'intelligence artificielle générative transforme profondément le développement logiciel.",
    "en": "Generative artificial intelligence is profoundly transforming software development.",
}


def count_tokens(tok, text):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


pd.DataFrame({
    name: {lang: count_tokens(tok, text) for lang, text in sentences.items()}
    for name, tok in {"Qwen3": tokenizer, "flan-t5": old_tokenizer}.items()
})

**Question** : pourquoi le français demande-t-il plus de tokens que l'anglais ? Quelles conséquences pratiques cela a-t-il (coût d'une API facturée au token, vitesse, fenêtre de contexte) ? Affichez aussi les tokens de la phrase française avec `flan-t5` : que remarquez-vous ?

✍️ **Votre réponse :**



### Le chat template

Les modèles conversationnels sont entraînés sur des conversations mises en forme avec des **tokens spéciaux** qui délimitent les rôles (`system`, `user`, `assistant`). Chaque famille de modèles a son propre format : le **chat template**, fourni avec le tokenizer. On ne l'écrit jamais à la main, on utilise `apply_chat_template`.

In [ ]:
messages = [
    {"role": "system", "content": "Tu es un assistant concis qui répond en français."},
    {"role": "user", "content": "Quelle est la capitale de la France ?"},
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,              # renvoie le texte plutôt que les ids
    add_generation_prompt=True,  # ajoute le début du tour de l'assistant
    enable_thinking=False,       # désactive le mode raisonnement (voir partie 4)
)
print(prompt_text)

**Question** : à quoi servent les tokens `<|im_start|>` et `<|im_end|>` ? Pourquoi le template se termine-t-il par `<|im_start|>assistant` ?

✍️ **Votre réponse :**



### La fenêtre de contexte

Un modèle ne peut traiter qu'un nombre limité de tokens à la fois : c'est sa **fenêtre de contexte**.

In [ ]:
print(f"flan-t5 : {old_tokenizer.model_max_length} tokens")
print(f"Qwen3   : {tokenizer.model_max_length} tokens")

del old_tokenizer

Gardez ces chiffres en tête : nous verrons dans la partie 4 que la limite de 512 tokens de `flan-t5` faussait une partie des conclusions des anciennes versions de ce TP.

## 2. Sous le capot : prédire le token suivant

Chargeons le modèle sur le GPU (environ 3,5 Go en float16).

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto")
print(f"Mémoire occupée par le modèle : {model.get_memory_footprint() / 1e9:.1f} Go")

Un LLM *causal* fait **une seule chose** : à partir d'une suite de tokens, il calcule un score (*logit*) pour chacun des ~150 000 tokens du vocabulaire. Un `softmax` transforme ces scores en probabilités pour le **token suivant**.

**Exercice 3** : compléter la fonction `next_token_distribution` qui renvoie le vecteur de probabilités du token suivant. Indications :
- `model(input_ids=...)` renvoie un objet dont l'attribut `logits` a la forme `(batch, longueur_séquence, taille_vocabulaire)` ;
- seuls les logits de la **dernière position** nous intéressent ;
- la température divise les logits **avant** le softmax.

In [ ]:
def next_token_distribution(text, temperature=1.0):
    """Renvoie le vecteur des probabilités du token suivant `text`."""
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


def plot_top_tokens(probs, k=10, title=""):
    """Affiche les k tokens les plus probables sous forme de diagramme en barres."""
    top = torch.topk(probs, k)
    labels = [repr(tokenizer.decode(i)) for i in top.indices]
    plt.barh(labels[::-1], top.values.cpu().numpy()[::-1])
    plt.xlabel("probabilité")
    plt.title(title)
    plt.show()


text = "La capitale de la France est"
plot_top_tokens(next_token_distribution(text), title=text)

**Exercice 4** : générer du texte, c'est appeler le modèle en boucle. Écrire la fonction `greedy_decode` qui, à chaque étape, choisit le token le plus probable (*greedy decoding*), l'ajoute à la séquence et recommence, jusqu'à `max_new_tokens` ou jusqu'au token de fin. Comparer avec le résultat de `model.generate(..., do_sample=False)`.

In [ ]:
EOS_IDS = model.generation_config.eos_token_id  # liste des ids qui terminent la génération


def greedy_decode(text, max_new_tokens=30):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


start = time.perf_counter()
print("Notre boucle    :", repr(greedy_decode(prompt_text)), f"({time.perf_counter() - start:.1f} s)")

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
start = time.perf_counter()
output_ids = model.generate(**inputs, max_new_tokens=30, do_sample=False)
reference = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("model.generate  :", repr(reference), f"({time.perf_counter() - start:.1f} s)")

**Question** : les deux sorties sont (normalement) identiques, mais `model.generate` est plus rapide. Notre boucle recalcule tout depuis le début à chaque token : quelle optimisation `generate` met-elle en place ? (indice : *KV cache*)

✍️ **Votre réponse :**



**Exercice 5** : visualiser la distribution du token suivant pour des températures de 0.2, 1.0 et 2.0. Comment la température modifie-t-elle la distribution ? Quelle valeur choisir pour une tâche factuelle ? Pour de l'écriture créative ?

In [ ]:
text = "Il était une fois un"

# VOTRE CODE ICI

✍️ **Votre réponse :**



## 3. Modèle *base* vs modèle *instruct*

Un LLM est construit en deux grandes étapes :
1. le **pré-entraînement** sur des milliers de milliards de tokens de texte brut : on obtient un modèle **base** qui sait compléter du texte ;
2. le **post-entraînement** (*instruction tuning*, puis apprentissage par renforcement) sur des conversations : on obtient un modèle **instruct** (ou *chat*) qui suit des consignes.

Comparons les deux versions de Qwen3-1.7B sur la même question.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(f"{MODEL_NAME}-Base", dtype=DTYPE, device_map="auto")

question = "Quelle est la capitale de la France ?"

inputs = tokenizer(question, return_tensors="pt").to(base_model.device)
output_ids = base_model.generate(**inputs, max_new_tokens=60, do_sample=False)
print("BASE, texte brut :\n", tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

chat_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print("\nINSTRUCT, chat template :\n", tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

# Libérons la mémoire GPU : nous n'utiliserons plus le modèle base.
del base_model
gc.collect()
torch.cuda.empty_cache()

**Question** : décrivez le comportement du modèle base. Pourquoi « continue »-t-il le texte au lieu de répondre ?

✍️ **Votre réponse :**



## 4. Prompt engineering

### Une fonction de génération réutilisable

Pour ne pas répéter le même code à chaque cellule, écrivons une fonction de génération.

**Exercice 6** : compléter la fonction `generate` qui :
- applique le chat template à la liste de `messages` (avec `add_generation_prompt=True`, `enable_thinking=enable_thinking`, `return_tensors="pt"` et `return_dict=True`) et place le résultat sur le device du modèle ;
- appelle `model.generate` en lui transmettant `max_new_tokens` et les paramètres supplémentaires `**gen_kwargs` ;
- renvoie uniquement le texte **généré** (sans le prompt), sans tokens spéciaux et sans espaces superflus.

In [ ]:
def generate(messages, max_new_tokens=256, enable_thinking=False, **gen_kwargs):
    """Génère la réponse de l'assistant à une liste de messages au format chat."""
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


# Vérification
answer = generate([{"role": "user", "content": "Quelle est la capitale de l'Italie ? Réponds en un mot."}], do_sample=False)
print(answer)
assert answer, "La réponse est vide."
assert "Réponds en un mot" not in answer, "La réponse contient le prompt : ne décodez que les tokens générés."
assert "<|im_end|>" not in answer, "La réponse contient des tokens spéciaux."

### Résumé de dialogues

Nous allons utiliser le LLM pour résumer des dialogues du dataset [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum), qui fournit pour chaque dialogue un résumé écrit par un humain.

In [ ]:
dataset = load_dataset("knkarthick/dialogsum")
dataset

In [ ]:
example_indices = [40, 200]
dash_line = "-" * 100

for index in example_indices:
    example = dataset["test"][index]
    print(dash_line)
    print(f"DIALOGUE {index} :\n{example['dialogue']}")
    print(dash_line)
    print(f"RÉSUMÉ HUMAIN :\n{example['summary']}\n")

Tentons une première approche : envoyer le dialogue seul, sans aucune consigne.

In [ ]:
for index in example_indices:
    dialogue = dataset["test"][index]["dialogue"]
    output = generate([{"role": "user", "content": dialogue}], max_new_tokens=100, do_sample=False)
    print(dash_line)
    print(f"RÉSUMÉ HUMAIN :\n{dataset['test'][index]['summary']}")
    print(f"\nSORTIE DU LLM SANS CONSIGNE :\n{output}\n")

**Question** : est-ce un bon résumé ? Pourquoi ?

✍️ **Votre réponse :**



### 4.1 Zero-shot

Ajoutons une consigne dans un **system prompt**, qui définit le rôle et le comportement de l'assistant pour toute la conversation. Fournir seulement une consigne, sans exemple, s'appelle l'inférence **zero-shot**.

In [ ]:
SYSTEM_PROMPT = (
    "You summarize dialogues. Write a short summary (one or two sentences) in English, "
    "in the third person, focusing on the key points."
)

for index in example_indices:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": dataset["test"][index]["dialogue"]},
    ]
    output = generate(messages, max_new_tokens=100, do_sample=False)
    print(dash_line)
    print(f"RÉSUMÉ HUMAIN :\n{dataset['test'][index]['summary']}")
    print(f"\nRÉSUMÉ ZERO-SHOT :\n{output}\n")

C'est bien mieux ! Mais le style ne correspond pas encore à celui des résumés humains (qui désignent par exemple les personnes par `#Person1#`). Montrons des exemples au modèle.

### 4.2 One-shot et few-shot

L'inférence **one-shot** et **few-shot** consiste à fournir au modèle un ou plusieurs exemples complets (dialogue, résumé) avant le dialogue à traiter. C'est l'**apprentissage en contexte** (*in-context learning*) : le modèle s'adapte à la tâche sans modifier ses poids.

Avec un modèle conversationnel, chaque exemple est représenté par un **tour de conversation** : un message `user` (le dialogue) suivi d'un message `assistant` (le résumé attendu).

Les exemples sont tirés du split `train` et l'évaluation se fait sur le split `test` : on ne doit jamais montrer au modèle les données sur lesquelles on l'évalue.

**Exercice 7** : compléter la fonction `make_messages` qui construit la liste de messages : le system prompt, puis un couple (`user`, `assistant`) par exemple de `shots`, puis le dialogue à résumer.

In [ ]:
def make_messages(dialogue, shots=(), system_prompt=SYSTEM_PROMPT):
    """Construit un prompt few-shot. `shots` est une liste d'exemples {"dialogue": ..., "summary": ...}."""
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


one_shot = dataset["train"].select([0])
messages = make_messages(dataset["test"][200]["dialogue"], one_shot)
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))

In [ ]:
few_shot = dataset["train"].select([0, 1, 2])

for name, shots in {"ONE-SHOT": one_shot, "FEW-SHOT (3 exemples)": few_shot}.items():
    output = generate(make_messages(dataset["test"][200]["dialogue"], shots), max_new_tokens=100, do_sample=False)
    print(dash_line)
    print(f"{name} :\n{output}")
print(dash_line)
print(f"RÉSUMÉ HUMAIN :\n{dataset['test'][200]['summary']}")

**Question** : combien de tokens compte le prompt few-shot ? Les anciennes versions de ce TP utilisaient `flan-t5` (fenêtre de 512 tokens) avec 3 exemples : que se passait-il ?

In [ ]:
# VOTRE CODE ICI

✍️ **Votre réponse :**



**Exercice 8** : modifier les exemples fournis (nombre, choix, exemples très différents du dialogue à résumer...) et le system prompt. Notez vos observations dans une cellule markdown. Les comparaisons à l'œil ont leurs limites : nous mesurerons objectivement l'apport du few-shot dans la partie 6.

✍️ **Votre réponse :**



### 4.3 Modèles de raisonnement (*thinking*)

Depuis fin 2024 (OpenAI o1, DeepSeek-R1...), les modèles de **raisonnement** produisent une longue réflexion avant de répondre : ils dépensent plus de calcul au moment de l'inférence (*test-time compute*) pour améliorer leurs réponses. Qwen3 est un modèle hybride : le paramètre `enable_thinking` du chat template active ou désactive ce mode. La réflexion est délimitée par les balises `<think>` et `</think>`.

Qwen recommande des paramètres d'échantillonnage différents selon le mode : nous les utilisons ci-dessous.

In [ ]:
puzzle = [{
    "role": "user",
    "content": "Alice a 3 frères. Chacun de ses frères a 2 sœurs. Combien de sœurs Alice a-t-elle ?",
}]

for enable_thinking in [False, True]:
    sampling = (
        dict(temperature=0.6, top_p=0.95, top_k=20) if enable_thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    )
    set_seed(0)
    start = time.perf_counter()
    output = generate(puzzle, max_new_tokens=2048, enable_thinking=enable_thinking, do_sample=True, **sampling)
    elapsed = time.perf_counter() - start

    print(dash_line)
    print(f"enable_thinking={enable_thinking} : {len(tokenizer(output)['input_ids'])} tokens générés en {elapsed:.1f} s")
    if enable_thinking and "</think>" not in output:
        print("\nRéflexion non terminée : max_new_tokens atteint.")
        continue
    thinking, _, answer = output.rpartition("</think>")
    if thinking:
        print(f"\nRÉFLEXION (début) :\n{thinking.replace('<think>', '').strip()[:800]}...")
    print(f"\nRÉPONSE :\n{answer.strip()}")

**Question** : comparez les réponses, le nombre de tokens générés et le temps de calcul. Relancez plusieurs fois (changez la graine) : les réponses sont-elles stables ? Dans quels cas le mode raisonnement vaut-il son coût ?

✍️ **Votre réponse :**



## 5. Influence des paramètres de génération

Au lieu de toujours choisir le token le plus probable (*greedy*), on peut **échantillonner** le token suivant dans la distribution calculée par le modèle. Plusieurs paramètres contrôlent cet échantillonnage :
- `do_sample` : active l'échantillonnage (sinon : greedy) ;
- `temperature` : aplatit ou accentue la distribution (vu à l'exercice 5) ;
- `top_k` : ne garde que les k tokens les plus probables ;
- `top_p` : ne garde que les tokens les plus probables dont la probabilité cumulée atteint p (*nucleus sampling*, p entre 0 et 1).

Documentation : [stratégies de génération](https://huggingface.co/docs/transformers/generation_strategies) et [GenerationConfig](https://huggingface.co/docs/transformers/main_classes/text_generation).

Notez que le modèle a ses propres valeurs par défaut, définies par ses auteurs :

In [ ]:
model.generation_config

**Exercice 9** : écrire une fonction `generate_summary(dialogue, temperature, top_k, top_p, do_sample, shots=few_shot)` qui renvoie le résumé généré avec ces paramètres (réutilisez `make_messages` et `generate`). Puis générer **3 résumés** du dialogue 200 pour chaque configuration de `configs`.

In [ ]:
configs = [
    dict(do_sample=False, temperature=None, top_k=None, top_p=None),  # greedy
    dict(do_sample=True, temperature=0.3, top_k=50, top_p=0.9),
    dict(do_sample=True, temperature=1.0, top_k=50, top_p=0.9),
    dict(do_sample=True, temperature=1.5, top_k=0, top_p=1.0),         # aucun filtrage
]

# VOTRE CODE ICI

**Exercice 10** : expliquer en 1 ou 2 lignes l'influence de chacun des 4 paramètres, en vous appuyant sur vos observations.

✍️ **Votre réponse :**



## 6. Évaluation automatique avec ROUGE

Jusqu'ici, nous avons jugé les résumés « à l'œil » sur 1 ou 2 exemples : ce n'est pas une évaluation. La métrique **ROUGE** mesure le recouvrement de mots entre un résumé généré et un résumé de référence :
- ROUGE-1 : recouvrement des mots ;
- ROUGE-2 : recouvrement des paires de mots consécutifs ;
- ROUGE-L : plus longue sous-séquence commune.

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
rouge.compute(
    predictions=["#Person1# asks #Person2# to send the report today."],
    references=["#Person1# wants #Person2# to send the report by the end of the day."],
)

Le split `test` de DialogSum fournit **3 résumés humains par dialogue** (identifiants `test_N_1`, `test_N_2`, `test_N_3`, sur 3 lignes consécutives). ROUGE accepte plusieurs références par prédiction : le score est calculé par rapport à la référence la plus proche.

**Exercice 11** : comparer les approches zero-shot, one-shot et few-shot sur 20 dialogues distincts du split `test` (en greedy, pour la reproductibilité). Construire un DataFrame avec une ligne par approche et une colonne par score ROUGE, puis l'afficher sous forme de diagramme en barres. Conservez les résumés few-shot dans une variable `few_shot_summaries` : ils serviront dans la partie 8.

Optimisation (bonus) : la génération un par un est lente. Comment générer plusieurs résumés en une seule passe ? (indice : *batching*, `padding_side="left"`)

In [ ]:
eval_set = dataset["test"].select(range(0, 60, 3))  # 20 dialogues distincts
eval_references = [dataset["test"][i:i + 3]["summary"] for i in range(0, 60, 3)]  # 3 références par dialogue

# VOTRE CODE ICI

**Question** : quelle approche obtient les meilleurs scores ? Citez au moins deux limites de ROUGE.

✍️ **Votre réponse :**



## 7. Sorties structurées

Dans une application, on a rarement besoin de texte libre : on veut des **données exploitables** (pour remplir une base de données, appeler une API...). On demande alors au LLM de répondre en **JSON** respectant un schéma, que l'on valide avec [Pydantic](https://docs.pydantic.dev/).

Définissons le schéma des informations à extraire d'un dialogue.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class DialogueInfo(BaseModel):
    topic: str = Field(description="Sujet principal du dialogue, en quelques mots")
    location: str = Field(description="Lieu où se déroule le dialogue, ou \"inconnu\" s'il n'est pas identifiable")
    action_items: list[str] = Field(description="Actions que les participants prévoient de faire")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="Ton général du dialogue")


def type_name(annotation):
    return annotation.__name__ if isinstance(annotation, type) else str(annotation).replace("typing.", "")


def describe_schema(schema):
    """Décrit les champs d'un modèle Pydantic, une ligne par champ (plus lisible qu'un schéma JSON pour un petit LLM)."""
    return "\n".join(
        f"- {name} ({type_name(field.annotation)}) : {field.description}" for name, field in schema.model_fields.items()
    )


print(describe_schema(DialogueInfo))

**Exercice 12** : compléter la fonction générique `generate_structured(messages, schema, max_retries=3)` qui :
1. ajoute au dernier message une consigne demandant de répondre uniquement en JSON avec les champs décrits par `describe_schema(schema)` ;
2. génère une réponse et extrait le JSON du texte (le modèle l'entoure parfois de balises Markdown) avec la fonction `extract_json` fournie ;
3. valide le JSON avec `schema.model_validate_json(...)`, qui lève une `ValidationError` en cas de problème ;
4. en cas d'erreur, ajoute la réponse fautive et le message d'erreur à la conversation, et redemande une correction au modèle, au plus `max_retries` fois.

In [ ]:
def extract_json(text):
    """Extrait le premier objet JSON {...} d'un texte."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else text


def generate_structured(messages, schema, max_retries=3, **gen_kwargs):
    """Génère une réponse validée par le modèle Pydantic `schema`."""
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


info = generate_structured(
    [{"role": "user", "content": f"Extrais les informations de ce dialogue :\n\n{dataset['test'][40]['dialogue']}"}],
    DialogueInfo,
    do_sample=False,
)
info

Appliquons l'extraction à plusieurs dialogues : nous obtenons directement une table, prête à être insérée dans une base de données.

In [ ]:
rows = [
    generate_structured(
        [{"role": "user", "content": f"Extrais les informations de ce dialogue :\n\n{dialogue}"}],
        DialogueInfo,
        do_sample=False,
    ).model_dump()
    for dialogue in eval_set.select(range(5))["dialogue"]
]
pd.DataFrame(rows)

**Remarque** : avec un petit modèle, donner le schéma JSON brut (`schema.model_json_schema()`) fonctionne mal : le modèle a tendance à recopier la structure du schéma (`{"properties": ...}`) au lieu de le remplir. Une description simple des champs donne de meilleurs résultats.

**Pour aller plus loin** : plutôt que de demander poliment au modèle de respecter le schéma et de réessayer en cas d'échec, les moteurs d'inférence actuels (vLLM, llama.cpp, Ollama, les API commerciales) proposent le **décodage contraint** : à chaque étape, les tokens qui rendraient le JSON invalide sont interdits (probabilité mise à zéro). La sortie est alors **garantie** conforme au schéma. Voir par exemple les bibliothèques [outlines](https://github.com/dottxt-ai/outlines) ou [xgrammar](https://github.com/mlc-ai/xgrammar).

## 8. LLM-as-a-judge

Une alternative à ROUGE : demander à un LLM d'évaluer les résumés selon une grille de critères. C'est aujourd'hui une pratique très répandue pour évaluer les applications à base de LLM.

**Exercice 13** : en utilisant `generate_structured`, écrire une fonction `judge(dialogue, summary)` qui renvoie un objet `Judgement` : une note de 1 à 5 évaluant la **fidélité** (pas d'information inventée ou fausse) et la **complétude** du résumé, accompagnée d'une justification. Appliquer cette fonction aux 20 résumés few-shot de la partie 6 et afficher la distribution des notes.

In [ ]:
class Judgement(BaseModel):
    # La justification est demandée AVANT la note : le modèle « raisonne » avant de conclure.
    justification: str = Field(description="Justification courte de la note")
    score: int = Field(ge=1, le=5, description="Note de 1 (mauvais) à 5 (excellent)")


JUDGE_PROMPT = """Tu évalues la qualité du résumé d'un dialogue.
Critères : fidélité (aucune information inventée ou fausse) et complétude (les points clés sont présents).

DIALOGUE :
{dialogue}

RÉSUMÉ À ÉVALUER :
{summary}"""


# VOTRE CODE ICI

**Question** : peut-on faire confiance à ce juge ? Citez des biais connus des LLM juges. Comment vérifier la fiabilité d'un juge ?

✍️ **Votre réponse :**



### Annexe : utiliser une API gratuite (optionnel)

Pour obtenir un juge plus puissant et **différent** du modèle évalué, vous pouvez utiliser une API offrant un accès gratuit (avec des limites de débit). La plupart sont compatibles avec le client `openai`, ce qui permet de changer de fournisseur en changeant seulement `base_url` et `model` :

| Fournisseur | Inscription | `base_url` |
|---|---|---|
| Google AI Studio (Gemini) | https://aistudio.google.com | `https://generativelanguage.googleapis.com/v1beta/openai/` |
| Groq | https://console.groq.com | `https://api.groq.com/openai/v1` |
| OpenRouter (modèles `:free`) | https://openrouter.ai | `https://openrouter.ai/api/v1` |
| Ollama (local, sur votre machine) | https://ollama.com | `http://localhost:11434/v1` |

Les noms des modèles et les quotas gratuits changent souvent : consultez la documentation du fournisseur (ou `client.models.list()`).

**Règles** :
- ne jamais écrire une clé d'API en clair dans un notebook : utilisez les **Secrets** de Colab (icône 🔑 dans la barre latérale), lus avec `userdata.get(...)` ;
- les offres gratuites peuvent utiliser vos données pour entraîner leurs modèles : n'envoyez **aucune donnée personnelle ou confidentielle**.

In [ ]:
# %pip install -q openai
# from google.colab import userdata
# from openai import OpenAI
#
# client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=userdata.get("GROQ_API_KEY"))
# response = client.chat.completions.create(
#     model="openai/gpt-oss-20b",  # à adapter : voir la liste des modèles du fournisseur
#     messages=[{"role": "user", "content": "Bonjour !"}],
# )
# print(response.choices[0].message.content)

## Bonus : quantification

Les poids d'un modèle sont stockés en float16 (2 octets par paramètre) : un modèle de 8 milliards de paramètres occupe donc environ 16 Go, trop pour le T4. La **quantification** stocke les poids sur moins de bits (8, 4, voire moins) au prix d'une légère perte de qualité. C'est ce qui permet d'exécuter des modèles sur un ordinateur portable (format GGUF avec llama.cpp, Ollama ou LM Studio).

La cellule ci-dessous charge [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) quantifié en 4 bits avec `bitsandbytes`. Elle télécharge 16 Go : exécutez-la seulement si vous avez le temps, après avoir libéré le modèle actuel.

**Question** : comparez la mémoire occupée et la qualité des résumés avec le modèle 1.7B.

In [ ]:
# %pip install -q bitsandbytes
# from transformers import BitsAndBytesConfig
#
# del model
# gc.collect()
# torch.cuda.empty_cache()
#
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
# )
# MODEL_NAME = "Qwen/Qwen3-8B"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization_config, device_map="auto")
# print(f"Mémoire occupée par le modèle : {model.get_memory_footprint() / 1e9:.1f} Go")

## Mini-projet (au choix)

**Durée estimée** : 3-4 heures. Choisissez **un** des deux sujets.

**Livrables attendus** (pour les deux sujets) :
- code documenté, factorisé (réutilisez `generate`, `generate_structured`...) ;
- un jeu de 10 questions de test avec les réponses attendues, et une évaluation de votre système sur ce jeu (exactitude, LLM-as-a-judge...) ;
- une analyse en markdown : ce qui fonctionne, ce qui échoue et pourquoi, pistes d'amélioration.

Pour de meilleurs résultats, vous pouvez utiliser un modèle plus gros qui tient aussi sur le T4, comme [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) (environ 8 Go en float16, sans mode raisonnement).

### Sujet A : questions-réponses sur un document (RAG)

Un LLM ne connaît pas vos documents, et sa fenêtre de contexte est limitée. Le **RAG** (*Retrieval-Augmented Generation*) consiste à découper les documents en morceaux (*chunks*), à **rechercher** les morceaux pertinents pour la question, puis à les fournir au LLM dans le prompt.

Réaliser une application qui :
1. charge un document PDF (téléversé par l'utilisateur ou téléchargé depuis une URL) et le découpe en chunks ;
2. calcule les *embeddings* des chunks avec un modèle d'embeddings (par exemple [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)) ;
3. **version « from scratch »** : pour une question, retrouve les chunks les plus proches (similarité cosinus) et génère une réponse qui cite ses sources ;
4. **version LangChain v1** : reproduit le même système avec [LangChain](https://docs.langchain.com/) (`PyPDFLoader`, `RecursiveCharacterTextSplitter`, `HuggingFaceEmbeddings`, un vector store, `ChatPromptTemplate`, `ChatHuggingFace`) ;
5. mémorise la conversation pour répondre aux questions de suivi (« Et qui sont ses auteurs ? »).

Votre application devra utiliser des templates de prompts.

### Sujet B : agent SQL avec appel d'outils

Un **agent** est un LLM qui peut appeler des **outils** (fonctions) en boucle pour accomplir une tâche : il décide quel outil appeler avec quels arguments, lit le résultat, puis continue jusqu'à pouvoir répondre.

Réaliser un agent qui répond à des questions en langage naturel sur une base de données SQLite :
1. créer une base SQLite (par exemple : étudiants, cours, notes) ;
2. écrire des outils Python documentés : obtenir le schéma de la base, exécuter une requête SQL **en lecture seule** ;
3. implémenter la boucle d'agent **à la main** avec l'appel d'outils natif de Qwen3 : `apply_chat_template(..., tools=[...])`, analyse des balises `<tool_call>`, exécution, renvoi du résultat dans un message de rôle `tool` ([documentation](https://huggingface.co/docs/transformers/chat_extras)) ;
4. tester des questions de difficulté croissante (« Combien y a-t-il d'étudiants ? », « Quel est le meilleur étudiant en BD ? », « Quelle est la moyenne par cours ? ») ;
5. (bonus) comparer avec un framework d'agents : [smolagents](https://huggingface.co/docs/smolagents) ou LangChain `create_agent`.

**Sécurité** : pourquoi la lecture seule est-elle indispensable ? Que pourrait faire un utilisateur malveillant sans cette protection (*prompt injection*) ?